In [24]:
import confnotebook

In [25]:
from pathlib import Path

source = Path("../examples/test/bag_15062026/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 000-591590 2024 Р±РЅ
[1] 000-591590 2025 Р±РЅ
[2] 000-591590 2026 Р±РЅ
[3] 04062001scan
[4] 1 квартал 2026
[5] 113130_Черно-бел. док-т_04062026
[6] 20260604scan150446
[7] 3 кв. 2025
[8] Scan_0034
[9] scan_20010116015111
[10] Акт сверки 01012026-31032026
[11] Акт сверки 2025 от 16.04.2026 (1)
[12] акт сверки 2025
[13] Акт сверки 2026 от 16.04.2026
[14] Акт сверки взаиморасчетов № 05800004074 от 29 мая 2026 г
[15] Акт сверки взаиморасчетов № 102 от 03 июня 2026 г
[16] Акт сверки взаиморасчетов № 151 от 20 мая 2026 г без 9_10 этапа
[17] Акт сверки взаиморасчетов № 6171 от 02 июня 2026 г
[18] Акт сверки май 2026
[19] Акт Сверки РУСАЛ Ачинск
[20] Акт сверки янв-мар 2026 (подпись К-ПМ)
[21] Акт сверки №1523 от 21.04.26 - 5f1bfca4-fde6-48ac-93cd-9266c8fa14d2
[22] Акт сверки №И--000238 от 06.04.26 РИТС
[23] Акт сверки
[24] Альфа-Металл
[25] АС АО КУРГАНМАШЗАВОД
[26] АС РБ за 2025 г
[27] ас скс 1кв 2026
[28] АС СУЭК апрель 26
[29] АСВ Лоста на 31.03.
[30] Байкал Аква 23-26
[31] БАЙКАЛ АКВА 

In [26]:
IDX_FILE = 5

In [27]:
import base64
import time

import requests

BASE_URL = "http://127.0.0.1:8001"
pdf_path = files[IDX_FILE]

document_b64 = base64.b64encode(pdf_path.read_bytes()).decode()
resp = requests.post(f"{BASE_URL}/send_reconciliation_act", json={"document": document_b64})

if not resp.ok:
    print(resp.text)


resp.raise_for_status()

process_id = resp.json()["process_id"]
print(f"process_id: {process_id}")

while True:
    resp = requests.post(f"{BASE_URL}/process_status", json={"process_id": process_id})
    if resp.status_code == 200:
        data = resp.json()
        print(f"seller: {data['seller']}")
        print(f"buyer:  {data['buyer']}")
        print(f"debit:  {len(data['debit'])} entries")
        print(f"credit: {len(data['credit'])} entries")
        break
    elif resp.status_code == 201:
        print("processing...")
        time.sleep(3)
    else:
        raise RuntimeError(resp.json())

process_id: b7fcb4e2-29ee-4ad3-8b0a-c3d8b6ca8573
processing...
processing...
processing...
processing...
processing...
processing...
processing...


RuntimeError: {'status': -2, 'message': "Не найдены колонки дебет/кредит: таблица '0'"}

In [ ]:
comments = """
            По данным АО "РУСАЛ Новокузнецк" на 30.09.2023
            задолженность в пользу АО "РУСАЛ Новокузнецк"
            составляет 13 755 023,24 руб.
            С разногласиями, протокол разногласий прилагается.
            Акт сверки проверен ОУФО ОЦО, ООО "РЦУ".
            Исполнитель: Воробьева Оксана Евгеньевна
            Дата:29.01.2025
            """

fill_resp = requests.post(f"{BASE_URL}/fill_reconciliation_act", json={
    "process_id": process_id,
    "comments": comments,
    "debit": data["debit"],
    "credit": data["credit"],
})
if not fill_resp.ok:
    print(fill_resp.json())
else:
    fill_resp.raise_for_status()

    out_path = f"../examples/output/{files[IDX_FILE].stem}_filled.pdf"
    Path(out_path).write_bytes(base64.b64decode(fill_resp.json()["document"]))
    print(out_path)

../examples/output/000-591590 2026 Р±РЅ_filled.pdf
